# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# もしものとき

**まず見る。そのまま残す。打つ手は人が選ぶ。** 事故の最中に自動で手を出すと、何が起きていたのかが消える。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

### 1. まず1本(`kmops` の権限で)

`host-emergency.sh` は**何も直さない・何も止めない。** コンテナ・健康状態・証明書の残り・ディスク・直近のエラーログを並べ、
最後に**そのまま貼れる復旧の手順**と控えの場所を出す。メールには頼らない(落ちているのは web かもしれない)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
./scripts/host-emergency.sh

### 1b. root で全部見る

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-emergency.sh"

### 1c. 同じ内容をファイルに残す

人に見てもらうときに添付できる。出力先を指定しなければ既定の場所に書く。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
./scripts/host-emergency.sh --collect /tmp/kosenmap-emergency.txt
ls -l /tmp/kosenmap-emergency.txt

## 2. 症状別の見どころ

| 症状 | まず見る | よくある原因 |
|---|---|---|
| **サイトが開かない** | コンテナの状態・`nginx -t`・証明書の残り | reverse-proxy が落ちた / 証明書切れ(**HSTS のため誰も入れない**) |
| **管理画面に入れない** | Logto のログ | **Logto が落ちると管理画面ごと入れない**(ゲートそのもの)。2026-08-28 は `.env` の打ち間違い(`KEY=="value"`)だった |
| **メールが来ない** | `.env` を compose が読めるか・送信サーバーの記録・cron の版と持ち主 | 合言葉に引用符(2026-09-13)/ cron.d が root 所有でない(2026-09-09) |
| 日曜の便りが来ない | ホストの控えの一覧・`backup.log` | 上に同じ。DB が落ちていると取れない |
| 「再起動が必要です」が続く | `/var/run/reboot-required` | 自動再起動は切ってある。**人が時期を選んで再起動する** |
| ディスクが足りない | `df -h /`・`docker system df` | 像の溜まり・ログ |
| **SSH で入れない** | 業者のコンソール | [../Old/docs/vps-setup.md](../Old/docs/vps-setup.md) §8「締め出されたときの逃げ道」 |

### compose が .env を読めるか

エラー文は出さない(値の断片が載る)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose config -q >/dev/null 2>&1; echo "docker compose config の終了コード: $?(0 なら読めている)"

### サービスのログ

`SERVICE` を見たいサービス名に書き換える(logto / web / reverse-proxy / mariadb / postgres / mailserver …)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
SERVICE=logto
docker compose logs --tail 100 "$SERVICE"

### 証明書の残り日数

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
echo | openssl s_client -connect ito4.jp:443 -servername ito4.jp 2>/dev/null | openssl x509 -noout -enddate

### ディスクの内訳

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
df -h /
echo
docker system df

## 3. 打つ手

**上で原因を見てから。** 「とりあえず再起動」は、何が起きていたかを消す。

### サービスを1つ再起動する

`SERVICE` を書き換える。**`docker compose down -v` は絶対に使わない**(ボリュームごと消える)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "指定したサービスを再起動します"
SERVICE=reverse-proxy
docker compose restart "$SERVICE"
docker compose ps

### 全体を立て直す

設定を変えていないのに止まったものを起こす。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "docker compose up -d で、止まっているコンテナを起こします" --timeout 900
docker compose up -d
docker compose ps

### ホストを再起動する

**会期中は避ける。** 再起動の間はサイトもメールも止まる。戻ったら [01-daily-check](01-daily-check.ipynb) を流す。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal --confirm "本番ホストを再起動します(数分サイトが止まります。sudo のパスワードを聞かれます)"
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo reboot"

## 4. データを戻すしかないとき

[03-backup](03-backup.ipynb) の §8(`Old/restore-data.ps1`。2026-09-14 に直し、使い捨ての環境で全部戻せることを確かめた)。
戻す前に、いまの状態も控えておく(壊れた状態でも、何が起きたかを後で調べられる)。